# K=11 Producao -- Orquestracao (execucao local ou Colab)

Pipeline de producao do modelo de regressao Bayesiana hierarquica K=11
(10 features baseline + mode_bin) para o produto **Diagnostico de Posicionamento**.

## Estrutura desta pasta (tudo num lugar so)

```
scripts/k11_pipeline/
├── 06_k11_pipeline.ipynb     # este notebook
├── train.py                  # treino NUTS K=11
├── evaluate.py               # metricas + asserts
├── export.py                 # exporta JSON para Next.js
└── spotify_tracks_limpo.parquet  # dataset
```

## Pre-requisitos

- **Python 3.10+**
- **GPU NVIDIA recomendada** (T4, RTX 3060+, A100). Sem GPU, o NUTS demora ~30h.
- Drivers CUDA 12.x (se for usar GPU local)
- No Colab: T4 gratuita (sessao de 12h)

## Setup rapido (Colab)

1. Faca upload da pasta `k11_pipeline/` inteira para o Colab (pode ser via zip)
2. Abra `06_k11_pipeline.ipynb` no Colab
3. Runtime > Change runtime type > T4 GPU
4. Run all cells

## Setup rapido (local)

```bash
cd /caminho/para/insights-spotfy-grupo-4/scripts/k11_pipeline
pip install -r ../../../requirements.txt  # ou o caminho equivalente
jupyter lab  # abre este notebook
```

## Arquitetura

- **Target:** `log(popularity + 1)` -> score 0-100 apos `exp() - 1`
- **Features (K=11):** danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, explicit, mode_bin
- **Hierarquica:** 107 generos, intercept + slopes especificos, prior nao-centrado
- **Sampler:** NUTS via NumPyro, 4 chains x 1000 draws, tune=1500, target_accept=0.9
  - GPU (T4/A100): ~3h
  - CPU so: ~30h
- **Validacao:** Train/Val/Test 70/15/15 com SEED=42, asserts RMSE<18, R2>0.30, HDI 0.90-0.97

## Etapas

1. `train.py` -- fit NUTS (~3h em GPU)
2. `evaluate.py` -- metricas em Val e Test, gera `q11_summary.json`
3. `export.py` -- converte NetCDF em JSON para o backend Next.js


In [1]:

# Instala o resto das dependencias
!pip install -q jax[cuda12] pymc==6.3.1 arviz==1.3.0 pytensor==3.3.0 numpyro==0.21.0 pandas pyarrow scipy scikit-learn

# Verifica backend do JAX
import jax
print()
print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
print('jax backend:', jax.default_backend())


jax version: 0.11.1
jax devices: [CudaDevice(id=0)]
jax backend: gpu


In [2]:
import os
import sys
from pathlib import Path

# Detecta se estamos no Colab
IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Jupyter local"}')
print()

PIPELINE_ROOT = None

if IN_COLAB:
    # Tenta montar Drive (silenciosamente se ja estiver montado)
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive', force_remount=False)
            print('Drive montado em /content/drive')
        else:
            print('Drive ja estava montado em /content/drive')
        print()
    except Exception as e:
        print(f'Drive nao disponivel (continuando sem): {e}')
        print()

    # Procura a pasta k11_pipeline (que contem train.py + parquet)
    candidates = [
        # Upload direto da pasta em /content/
        Path('/content/k11_pipeline'),
        # Se subiu apenas o zip, o usuario pode ter extraido em qualquer lugar
        # Dentro do Drive
        Path('/content/drive/MyDrive/k11_pipeline'),
        Path('/content/drive/MyDrive/insights-spotfy-grupo-4/scripts/k11_pipeline'),
    ]

    found = None
    for c in candidates:
        if c.is_dir() and (c / 'train.py').exists() and (c / 'spotify_tracks_limpo.parquet').exists():
            found = c
            break

    # Fallback: busca recursiva por train.py com parquet no mesmo dir
    if found is None:
        print('Procurando recursivamente em /content/...')
        for p in Path('/content').rglob('train.py'):
            if (p.parent / 'spotify_tracks_limpo.parquet').exists():
                found = p.parent
                break

    if found is None:
        print('!!! Pasta k11_pipeline nao encontrada.')
        print()
        print('Como subir (escolha UM):')
        print()
        print('Opcao A (mais rapida, ~30s): upload direto da pasta')
        print('  1. Arraste a pasta k11_pipeline/ para o painel Files (esquerda)')
        print('  2. Ela sera salva em /content/k11_pipeline/')
        print('  3. Re-rode esta celula')
        print()
        print('Opcao B (persiste entre sessoes): coloque no Google Drive')
        print('  1. Upload da pasta k11_pipeline/ para /content/drive/MyDrive/k11_pipeline/')
        print('  2. Re-rode esta celula')
        print()
        print('Opcao C (zip): faca upload do zip, depois rode:')
        print('  !unzip k11_pipeline.zip -d /content/')
        raise FileNotFoundError('k11_pipeline nao encontrada')

    PIPELINE_ROOT = found
    print(f'k11_pipeline encontrada: {PIPELINE_ROOT}')
    print()

else:
    # Jupyter local: usa o diretorio do proprio notebook
    PIPELINE_ROOT = Path(os.getcwd())
    if not (PIPELINE_ROOT / 'train.py').exists():
        # Tenta subir niveis
        for _ in range(5):
            PIPELINE_ROOT = PIPELINE_ROOT.parent
            if (PIPELINE_ROOT / 'train.py').exists():
                break
    print(f'PIPELINE_ROOT: {PIPELINE_ROOT}')
    print()

os.chdir(PIPELINE_ROOT)

# Garante diretorios que os scripts escrevem
(PIPELINE_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
(PIPELINE_ROOT / 'relatorio' / 'analises' / 'resultados').mkdir(parents=True, exist_ok=True)

# Verificacoes finais
print('=== Estrutura ===')
for f in ['train.py', 'evaluate.py', 'export.py', 'spotify_tracks_limpo.parquet', 'artifacts/', 'relatorio/analises/resultados/']:
    full = PIPELINE_ROOT / f
    status = '[OK]' if full.exists() else '[FALTA]'
    print(f'  {status}  {f}')

# Assertions
assert (PIPELINE_ROOT / 'train.py').exists(), 'train.py nao encontrado'
assert (PIPELINE_ROOT / 'evaluate.py').exists(), 'evaluate.py nao encontrado'
assert (PIPELINE_ROOT / 'export.py').exists(), 'export.py nao encontrado'
assert (PIPELINE_ROOT / 'spotify_tracks_limpo.parquet').exists(), 'parquet nao encontrado'
print()
print('Tudo certo. Pode prosseguir para C4.')


Ambiente: Google Colab

Drive ja estava montado em /content/drive

k11_pipeline encontrada: /content/drive/MyDrive/k11_pipeline

=== Estrutura ===
  [OK]  train.py
  [OK]  evaluate.py
  [OK]  export.py
  [OK]  spotify_tracks_limpo.parquet
  [OK]  artifacts/
  [OK]  relatorio/analises/resultados/

Tudo certo. Pode prosseguir para C4.


In [3]:
# === SMOKE TEST: valida pipeline end-to-end em ~1-2 min ===
# Usa config minima (10 draws, 10 tune, 2 chains) e escreve em artifacts/smoke/.
# Se esta celula terminar com [OK] Smoke test passou, o pipeline esta OK para o fit real.
# Se explodir, veja o erro abaixo e NAO rode C5 (fit longo) ate resolver.
import shutil
from pathlib import Path

smoke_dir = Path('artifacts/smoke')
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
smoke_dir.mkdir(parents=True)

print()
print('=== Smoke test (10/10/2 -> artifacts/smoke/) ===')
print()
!python train.py --draws 10 --tune 10 --chains 2 --out-dir artifacts/smoke 2>&1 | tee smoke.log

# Verifica checkpoints foram salvos
smoke_artifacts = Path('artifacts/smoke')
required = ['k11_posterior.nc', 'scaler.json', 'feature_names.json', 'genero_cats.json', 'split_indices.npz']
missing = [f for f in required if not (smoke_artifacts / f).exists()]

print()
if missing:
    print('[ERRO] Smoke test falhou -- faltam:', missing)
    print('NAO rode C5 ate resolver. Veja smoke.log acima.')
    raise SystemExit(1)
else:
    print('[OK] Smoke test passou -- todos os artefatos foram salvos.')
    print('  Posterior:', (smoke_artifacts / 'k11_posterior.nc').stat().st_size // 1024, 'KB')
    print('  Scaler:', (smoke_artifacts / 'scaler.json').stat().st_size, 'bytes')
    print('  Pode prosseguir para C5 (fit real, ~3h em T4 ou ~30min em CPU 4-core).')
    # Limpa o smoke -- C5 vai reescrever em artifacts/ (sem /smoke)
    shutil.rmtree(smoke_dir)
    if Path('smoke.log').exists():
        Path('smoke.log').unlink()



=== Smoke test (10/10/2 -> artifacts/smoke/) ===

train_k11 — Modelo Bayesiano Hierárquico K=11
SEED=42 | K=11 | jax backend=gpu | devices=1 | draws=10 tune=10 chains=2 | out=smoke
[1/8] Lendo spotify_tracks_limpo.parquet ...
      Removidos 5969 faixas não-musicais. Restantes: 83,771
[2/8] Construindo K=11 e dropna ...
      dropna: 19 removidos. Restantes: 83,752
[3/8] Z-score + log1p(target) ...
      X.shape=(83752, 11), y_log.shape=(83752,)
      #generos: 107
      [checkpoint] scaler/feature_names/genero_cats salvos em artifacts/smoke
[4/8] Split 70/15/15 ...
      train=58,626 | val=12,562 | test=12,564
[5/8] Construindo modelo PyMC ...
      coords: genero=107, feature=11
[6/8] Ajustando NUTS (numpyro, 2 chains, draws=10, tune=10) ...
Only 10 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
NUTS[numpyro]: [mu_alpha, sigma_alpha, mu_beta, sigma_beta, z_alpha, z_beta, sigma_y]
/usr/local/lib/python3.13/dist-packages/pymc/samplin

SystemExit: 1

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:

!python train.py 2>&1 | tee train.log


train_k11 — Modelo Bayesiano Hierárquico K=11
SEED=42 | K=11 | jax backend=gpu | devices=1 | draws=1000 tune=1500 chains=4 | out=artifacts
[1/8] Lendo spotify_tracks_limpo.parquet ...
      Removidos 5969 faixas não-musicais. Restantes: 83,771
[2/8] Construindo K=11 e dropna ...
      dropna: 19 removidos. Restantes: 83,752
[3/8] Z-score + log1p(target) ...
      X.shape=(83752, 11), y_log.shape=(83752,)
      #generos: 107
      [checkpoint] scaler/feature_names/genero_cats salvos em /content/drive/MyDrive/k11_pipeline/artifacts
[4/8] Split 70/15/15 ...
      train=58,626 | val=12,562 | test=12,564
[5/8] Construindo modelo PyMC ...
      coords: genero=107, feature=11
[6/8] Ajustando NUTS (numpyro, 4 chains, draws=1000, tune=1500) ...
NUTS[numpyro]: [mu_alpha, sigma_alpha, mu_beta, sigma_beta, z_alpha, z_beta, sigma_y]
/usr/local/lib/python3.13/dist-packages/pymc/sampling/jax.py:463: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be

In [5]:
!python evaluate.py 2>&1 | tee evaluate.log


evaluate_k11 — Avaliação offline K=11 (Val + Test)
SEED=42 | n_samples=1000 | HDI=94% | artifacts=artifacts
[1/6] Lendo spotify_tracks_limpo.parquet ...
      Removidos 5969 faixas não-musicais. Restantes: 83,771
[2/6] Construindo K=11 e dropna ...
      dropna: 19 removidos. Restantes: 83,752
[3/6] Aplicando scaler de artifacts/scaler.json ...
      X.shape=(83752, 11)
      splits: train=58,626 | val=12,562 | test=12,564
[4/6] Carregando posterior artifacts/k11_posterior.nc ...
      posterior sub-amostrado: 1000/4000
[5/6] Predição em Val e Test ...
      Val  RMSE=19.471  MAE=12.848  R²=0.133  log-RMSE=0.995  HDI94=0.392
      Test RMSE=19.123  MAE=12.743  R²=0.152  log-RMSE=0.981  HDI94=0.400
      [checkpoint] relatorio/analises/resultados/q11_val_metrics.csv
      [checkpoint] relatorio/analises/resultados/q11_test_metrics.csv
      Per-gênero (Test, top 10):
        forro                    n=  162  RMSE=4.024  MAE=3.147
        bluegrass                n=  159  RMSE=8.457  MAE

In [6]:
!python export.py 2>&1 | tee export.log


export_for_nextjs — Posterior -> JSON para Next.js
SEED=42 | N_SAMPLES=1000 | artifacts=/content/drive/MyDrive/k11_pipeline/artifacts
[1/4] Lendo artifacts/k11_posterior.nc ...
      chains=4 | draws=1000 | total=4000
      feature_names=11 | genero_cats=107
[2/4] Calculando sumário ...
[3/4] Empilhando chains e sub-amostrando 1000 samples ...
Traceback (most recent call last):
  File "/content/drive/MyDrive/k11_pipeline/export.py", line 377, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/k11_pipeline/export.py", line 354, in main
    samples = build_samples(idata)
  File "/content/drive/MyDrive/k11_pipeline/export.py", line 271, in build_samples
    stacked = post.stack(sample=("chain", "draw"))  # dims: sample, ...
              ^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/xarray/core/common.py", line 306, in __getattr__
    raise AttributeError(
        f"{type(self).__name__!r} object has no attribute {name!r}"
    )
AttributeError: 'DataTree' object

In [8]:
import re

# Path to the script
export_script = 'export.py'

with open(export_script, 'r') as f:
    content = f.read()

# The problematic line is likely: stacked = post.stack(sample=("chain", "draw"))
# We need to make sure 'post' is a Dataset, not a DataTree.
# Usually, 'post' comes from 'idata.posterior'.

# We will inject a check to convert DataTree to Dataset if necessary
old_line = 'stacked = post.stack(sample=("chain", "draw"))'
new_line = """if hasattr(post, 'to_dataset'):
        post = post.to_dataset()
    stacked = post.stack(sample=(\"chain\", \"draw\"))"""

if old_line in content:
    new_content = content.replace(old_line, new_line)
    with open(export_script, 'w') as f:
        f.write(new_content)
    print(f'Successfully patched {export_script}')
else:
    print(f'Could not find the target line in {export_script}. Please check the file manually.')

Successfully patched export.py


In [9]:
# Retry the export after patching
!python export.py 2>&1 | tee export_patched.log

export_for_nextjs — Posterior -> JSON para Next.js
SEED=42 | N_SAMPLES=1000 | artifacts=/content/drive/MyDrive/k11_pipeline/artifacts
[1/4] Lendo artifacts/k11_posterior.nc ...
      chains=4 | draws=1000 | total=4000
      feature_names=11 | genero_cats=107
[2/4] Calculando sumário ...
[3/4] Empilhando chains e sub-amostrando 1000 samples ...
      total=4000 -> selecionados=1000
[4/4] Escrevendo artefatos ...
      - artifacts/k11_posterior_summary.json
      - artifacts/k11_posterior_samples.json.gz

ARTEFATOS GERADOS
  k11_posterior_summary.json                 56.12 KB
  k11_posterior_samples.json.gz           11908.63 KB
  scaler.json                                 0.75 KB
  feature_names.json                          0.17 KB
  genero_cats.json                            1.37 KB


In [7]:
import json
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

print('=== Artefatos gerados ===\n')
for f in sorted(Path('artifacts').iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:8.1f} KB')

print('\n=== Metricas ===\n')
summary_path = Path('relatorio/analises/resultados/q11_summary.json')
assertions_passed = False
if summary_path.exists():
    with open(summary_path) as fh:
        summary = json.load(fh)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    # Chave flat (top-level) -- escrita por evaluate.py
    assertions_passed = summary.get('assertions_passed', False)
    print('\n[OK] assertions_passed:', assertions_passed)
else:
    print('AVISO: q11_summary.json nao encontrado -- verifique se evaluate.py rodou sem erro.')

# Secao de download para Colab
if IN_COLAB and assertions_passed:
    print('\n=== Download dos artefatos (Colab) ===\n')
    print('Os artefatos estao em:', Path('artifacts').resolve())
    print()
    print('Para usar no backend Next.js local, copie de volta para o repo:')
    print('  scripts/k11_pipeline/artifacts/*.json  ->  artifacts/  (raiz do repo)')
    print('  scripts/k11_pipeline/artifacts/*.gz    ->  artifacts/  (raiz do repo)')
    print()
    print('Comandos equivalentes no PowerShell local:')
    print('  Copy-Item scripts/k11_pipeline/artifacts/* -Destination artifacts/ -Force')
    print()
    print('Opcao 1 (manual): baixe um por um pelo painel Files (botao direito > Download)')
    print()
    print('Opcao 2 (zip automatico): disparando download abaixo...')
    print()
    # Dispara download do zip
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive('k11_artifacts', 'zip', 'artifacts')
    files.download(zip_path)


=== Artefatos gerados ===

  feature_names.json                                  0.2 KB
  genero_cats.json                                    1.4 KB
  k11_posterior.nc                                88375.4 KB
  scaler.json                                         0.7 KB
  smoke                                               4.0 KB
  split_indices.npz                                 655.1 KB

=== Metricas ===

{
  "val": {
    "rmse": 19.47123146057129,
    "mae": 12.847837448120117,
    "r2": 0.13300701823817151,
    "log_rmse": 0.9951373934745789,
    "hdi_94_coverage": 0.39245343098232766
  },
  "test": {
    "rmse": 19.122812271118164,
    "mae": 12.743186950683594,
    "r2": 0.15207769596272291,
    "log_rmse": 0.9812955260276794,
    "hdi_94_coverage": 0.4004297994269341
  },
  "per_genre_top10_test": [
    {
      "genero": "forro",
      "n": 162,
      "rmse": 4.023683071136475,
      "mae": 3.146993637084961
    },
    {
      "genero": "bluegrass",
      "n": 159,
      "rmse"

## Proximos passos

### Se `assertions_passed == true:`

1. **Copiar artefatos para o repo (para o backend Next.js):**

   O backend Next.js espera os artefatos em `<repo_root>/artifacts/`, nao em `scripts/k11_pipeline/artifacts/`.

   No PowerShell, na raiz do repo:
   ```powershell
   Copy-Item scripts/k11_pipeline/artifacts/* -Destination artifacts/ -Force
   ```

   Ou no bash:
   ```bash
   cp scripts/k11_pipeline/artifacts/* artifacts/
   ```

2. **Commitar (opcional):**
   ```bash
   git add scripts/k11_pipeline/ artifacts/ scripts/k11_pipeline/*.log
   git commit -m "feat: K=11 modelo treinado e validado"
   git push
   ```

3. **Subir o backend Next.js (em outra pasta ou outra maquina):**
   ```bash
   cd /caminho/para/insights-spotfy-grupo-4
   # garantir que os artefatos estao em ./artifacts/
   npm install
   cp .env.local.example .env.local
   # editar .env.local e colocar OPENROUTER_API_KEY=sk-or-v1-...
   npm run dev
   ```

4. **Testar o endpoint:**
   ```bash
   curl -X POST http://localhost:3000/api/diagnose \
     -H "Content-Type: application/json" \
     -d '{
       "track_features": {
         "danceability": 0.7, "energy": 0.5, "loudness": -5.0,
         "speechiness": 0.05, "acousticness": 0.3,
         "instrumentalness": 0.0, "liveness": 0.1,
         "valence": 0.6, "tempo": 120.0, "explicit": 0, "mode_bin": 0
       },
       "genero": "sertanejo"
     }'
   ```

### Se `assertions_passed == false:`

Investigar `q11_summary.json` e ver qual metrica falhou:

| Metrica | Falha comum | Acao |
|---------|-------------|------|
| RMSE >= 18 | Modelo nao captura variancia | Aumentar K? (nao recomendado, Q8 v2 mostrou overfit) |
| R2 <= 0.30 | Pouca variancia explicada | Aceitar -- pode ser teto do problema |
| HDI fora de [0.90, 0.97] | Calibracao ruim | Ajustar priors sigma_alpha/sigma_beta |

## Caveats do modelo

- **Genero deve ser conhecido** -- o dropdown tem 107 opcoes apos filtro nao-musical
- **Score e preditivo, nao causal** -- diz "o que costuma acontecer", nao "como fazer hit"
- **Calibrado em popularity do Spotify (0-100)**, nao em qualidade musical
- **NUTS aproxima o posterior** -- HDI e uma estimativa, nao certeza

## Se algo der errado

Logs ficam salvos em:
- `train.log` -- log completo do treino (inclui R-hat, ESS, divergencias)
- `evaluate.log` -- log da avaliacao
- `export.log` -- log do export

Para debug, rode os scripts diretamente no terminal (cwd = scripts/k11_pipeline/):
```bash
python train.py
```
e veja o erro com traceback completo.
